# 99.5 Advanced Optimization: Exercises

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Method calls, keyword arguments and component indices.

**Practises:** fitting binding parameters to several experiments with one shared parameter set, one objective per experiment and component-wise variables.

1. Fit binding parameters to three experiments

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

```{note}
**Optional material.** Chapter 99 is not part of the main sequence. Work through it after the related main topic or whenever time permits.
```

Binding parameters are fitted in the same way as transport parameters in **04.2 Parameter Fitting**: a reference and a comparator describe each measurement, variables point to model parameters and the simulator is registered as an evaluator. The difference is that one experiment is not enough. This exercise extends **04.2 Parameter Fitting: Exercises**, Exercise 4, where the three binding experiments were compared with the initial estimates.

## Starter: three binding experiments

The column and tubing transport are fixed from the tracer fits. Three experiments constrain the `StericMassAction` parameters of proteins **A, B and C**: a breakthrough and two salt gradients of **15 and 120 column volumes (CV)**. Each workbook contains **Salt, A, B and C** in that order.

| Setting | Salt, A, B, C |
|---|---|
| `adsorption_rate` | `[0, 2.3e-5, 0.559, 3.5e-3]` |
| `desorption_rate` | `[0, 1, 1, 1]` |
| `characteristic_charge` | `[1, 6.9, 2.3, 5.8]` (salt valence first) |
| `steric_factor` | `[0, 10, 10.6, 11.83]` |
| `capacity` | `1200` (single value) |

The [setup script](../../04%20Characterization%20and%20Optimization/resources/parameter_fitting_exercises_setup.py) builds the three processes.

```{tip}
Run the cell below once with **Shift+Enter** to create the processes, load the measurements and set up one comparator per experiment. The 120 CV gradient takes a little longer to simulate.
```

In [ ]:
%matplotlib inline
import pandas as pd
from CADETProcess.processModel import ComponentSystem
from CADETProcess.simulator import Cadet
from CADETProcess.reference import ReferenceIO
from CADETProcess.comparison import Comparator, Shape
from CADETProcess.optimization import OptimizationProblem, U_NSGA3

%run "../../04 Characterization and Optimization/resources/parameter_fitting_exercises_setup.py"
data_folder = "../../04 Characterization and Optimization/04.2 Parameter Fitting/experimental_data"

component_system = ComponentSystem(["Salt", "A", "B", "C"])
process_bt = breakthrough_process(component_system)
process_15cv = gradient_process(15, component_system)
process_120cv = gradient_process(120, component_system)

data_bt = pd.read_excel(f"{data_folder}/breakthrough.xlsx", index_col=0)
data_15cv = pd.read_excel(f"{data_folder}/15cv_gradient.xlsx", index_col=0)
data_120cv = pd.read_excel(f"{data_folder}/120cv_gradient.xlsx", index_col=0)

reference_bt = ReferenceIO("experiment bt", data_bt.index, data_bt, component_system=component_system)
reference_15cv = ReferenceIO("experiment 15cv", data_15cv.index, data_15cv, component_system=component_system)
reference_120cv = ReferenceIO("experiment 120cv", data_120cv.index, data_120cv, component_system=component_system)

# One comparator per experiment, with one Shape metric per protein at the UV detector.
comparator_bt = Comparator(name=reference_bt.name)
comparator_bt.add_difference_metric(Shape(reference_bt, components=["A"]), "pipe6.outlet")
comparator_bt.add_difference_metric(Shape(reference_bt, components=["B"]), "pipe6.outlet")
comparator_bt.add_difference_metric(Shape(reference_bt, components=["C"]), "pipe6.outlet")
comparator_15cv = Comparator(name=reference_15cv.name)
comparator_15cv.add_difference_metric(Shape(reference_15cv, components=["A"]), "pipe6.outlet")
comparator_15cv.add_difference_metric(Shape(reference_15cv, components=["B"]), "pipe6.outlet")
comparator_15cv.add_difference_metric(Shape(reference_15cv, components=["C"]), "pipe6.outlet")
comparator_120cv = Comparator(name=reference_120cv.name)
comparator_120cv.add_difference_metric(Shape(reference_120cv, components=["A"]), "pipe6.outlet")
comparator_120cv.add_difference_metric(Shape(reference_120cv, components=["B"]), "pipe6.outlet")
comparator_120cv.add_difference_metric(Shape(reference_120cv, components=["C"]), "pipe6.outlet")

simulator = Cadet()
simulator.time_resolution = 0.5

## Exercise 1: fit binding parameters to three experiments

One shared set of binding parameters must describe all three experiments, but each experiment keeps its own comparator.

**Step 1:** create an `OptimizationProblem` named `binding_problem`, register the simulator as an evaluator, then register each process and add its comparator as an objective that scores **only that process**:

| Process | Comparator | Objective name |
|---|---|---|
| `process_bt` | `comparator_bt` | `"binding_bt_error"` |
| `process_15cv` | `comparator_15cv` | `"binding_15cv_error"` |
| `process_120cv` | `comparator_120cv` | `"binding_120cv_error"` |

```{tip}
**Methods for this exercise**

- `add_evaluator(simulator)` and `add_evaluation_object(process_bt)`, as in 04.2.
- `add_objective(comparator_bt, name="binding_bt_error", n_objectives=comparator_bt.n_metrics, evaluation_objects=[process_bt], requires=[simulator])`: `evaluation_objects` restricts the objective to that process. See the [evaluator guide](https://cadet-process.readthedocs.io/en/latest/user_guide/optimization/evaluator.html).
```

In [ ]:
# Problem with one objective per experiment
binding_problem = OptimizationProblem("SMA_binding")
binding_problem.add_evaluator(simulator)

binding_problem.add_evaluation_object(process_bt)
binding_problem.add_objective(
    comparator_bt, name="binding_bt_error", n_objectives=comparator_bt.n_metrics,
    evaluation_objects=[process_bt], requires=[simulator],
)
binding_problem.add_evaluation_object(process_15cv)
binding_problem.add_objective(
    comparator_15cv, name="binding_15cv_error", n_objectives=comparator_15cv.n_metrics,
    evaluation_objects=[process_15cv], requires=[simulator],
)
binding_problem.add_evaluation_object(process_120cv)
binding_problem.add_objective(
    comparator_120cv, name="binding_120cv_error", n_objectives=comparator_120cv.n_metrics,
    evaluation_objects=[process_120cv], requires=[simulator],
)

**Step 2:** add the three binding variables for **protein A**. Component lists start with salt, so A has index **1**. Without `targets`, a variable changes all registered processes, so the three experiments share the value.

| Variable name | Parameter path | Lower bound | Upper bound | Normalization |
|---|---|---:|---:|---|
| `adsorption_rate_1` | `flow_sheet.column.binding_model.adsorption_rate` | `1e-7` | `1` | `"auto"` |
| `characteristic_charge_1` | `flow_sheet.column.binding_model.characteristic_charge` | `1` | `10` | `"linear"` |
| `steric_factor_1` | `flow_sheet.column.binding_model.steric_factor` | `1` | `30` | `"auto"` |

With `desorption_rate` fixed at 1, the fitted `adsorption_rate` equals the equilibrium constant.

```{tip}
`add_variable(name, parameter_path=..., lb=..., ub=..., normalization=..., indices=[1])` changes one entry of a component list. See [variable indices](https://cadet-process.readthedocs.io/en/latest/user_guide/optimization/variable_indices.html).
```

In [ ]:
# Binding variables of protein A (index 1)
binding_problem.add_variable(
    "adsorption_rate_1", parameter_path="flow_sheet.column.binding_model.adsorption_rate",
    lb=1e-7, ub=1, normalization="auto", indices=[1],
)
binding_problem.add_variable(
    "characteristic_charge_1", parameter_path="flow_sheet.column.binding_model.characteristic_charge",
    lb=1, ub=10, normalization="linear", indices=[1],
)
binding_problem.add_variable(
    "steric_factor_1", parameter_path="flow_sheet.column.binding_model.steric_factor",
    lb=1, ub=30, normalization="auto", indices=[1],
)

The supplied cell adds the same three variables for B (index 2) and C (index 3) and checks the problem: nine variables and three objectives with three metrics each.

In [ ]:
# The same variables for proteins B and C.
binding_problem.add_variable(
    "adsorption_rate_2", parameter_path="flow_sheet.column.binding_model.adsorption_rate",
    lb=1e-7, ub=1, normalization="auto", indices=[2],
)
binding_problem.add_variable(
    "characteristic_charge_2", parameter_path="flow_sheet.column.binding_model.characteristic_charge",
    lb=1, ub=10, normalization="linear", indices=[2],
)
binding_problem.add_variable(
    "steric_factor_2", parameter_path="flow_sheet.column.binding_model.steric_factor",
    lb=1, ub=30, normalization="auto", indices=[2],
)
binding_problem.add_variable(
    "adsorption_rate_3", parameter_path="flow_sheet.column.binding_model.adsorption_rate",
    lb=1e-7, ub=1, normalization="auto", indices=[3],
)
binding_problem.add_variable(
    "characteristic_charge_3", parameter_path="flow_sheet.column.binding_model.characteristic_charge",
    lb=1, ub=10, normalization="linear", indices=[3],
)
binding_problem.add_variable(
    "steric_factor_3", parameter_path="flow_sheet.column.binding_model.steric_factor",
    lb=1, ub=30, normalization="auto", indices=[3],
)

print(binding_problem.variable_names)
print(binding_problem.n_objectives)

**Optional:** the search uses `U_NSGA3` with **64 candidates**, at most **12 generations** and **8 cores**. It takes a long time; uncomment the `optimize()` call to run it and reduce `n_cores` for a smaller machine.

In [ ]:
# Optional search: uncomment the last lines to run it.
binding_optimizer = U_NSGA3()
binding_optimizer.n_cores = 8
binding_optimizer.pop_size = 64
binding_optimizer.n_max_iter = 12

# binding_results = binding_optimizer.optimize(binding_problem, use_checkpoint=False)
# print(binding_results.x)

Why does this fit need all three experiments, and what should be checked after the search?

```{dropdown} Answer
The experiments contain different information: the breakthrough mainly constrains capacity and steric shielding, while two gradients of different length separate the salt dependence (characteristic charge) from the binding strength. One experiment alone would leave several parameter combinations with the same curve.

After the search, the three comparison plots should be inspected separately: a good overall score can hide a poor match in one experiment. With only 12 generations, the search is a demonstration; parameters at their bounds call for a longer run before the values are relied on.
```